<a href="https://colab.research.google.com/github/Tech-Matt/tiny-transformers/blob/main/transformer_from_scratch_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Input Embedding

In [2]:
import torch
import torch.nn as nn
import math

In [3]:
class InputEmbeddings(nn.Module):
    """
    Input Embeddings — translates token IDs (plain integers) into rich
    vectors of numbers that carry semantic meaning.

    Why we need it: the transformer cannot work with raw words or even
    raw integers. It needs every token to be represented as a vector of
    numbers (an embedding) so that mathematical operations like attention
    can find relationships between words.

    How it works: internally nn.Embedding is just a lookup table — a big
    matrix of shape (vocab_size, d_model). Each row is one word's vector.
    Given a token ID, it simply returns that row. The values start random
    but are gradually learned during training so that words with similar
    meanings end up with similar vectors.

    Example with d_model=4:
        token ID 0 "the"  →  [ 0.21, -0.54,  0.87,  0.13]
        token ID 1 "cat"  →  [ 0.92, -0.11,  0.34,  0.67]
        token ID 2 "sat"  →  [-0.45,  0.78,  0.02, -0.33]
    """

    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()

        # d_model: how many numbers represent each word (e.g. 512).
        # A larger d_model means more capacity to encode meaning,
        # but also more computation. We store it because we need it
        # in forward() for the scaling step.
        self.d_model = d_model

        # vocab_size: how many unique tokens exist in our vocabulary.
        # This determines the number of rows in the lookup table —
        # one row per possible token.
        self.vocab_size = vocab_size

        # nn.Embedding creates the lookup table of shape (vocab_size, d_m

## 2. Positional Encoding

In [4]:
class PositionalEncoding(nn.Module):
    """
    Positional Encoding — injects information about token position into
    the word embeddings.

    Why we need it: the transformer processes all tokens simultaneously
    (in parallel), so unlike an RNN it has no built-in sense of order.
    "The cat sat" and "sat the cat" would look identical without this.

    How it works: we compute a fixed vector of sin/cos values for each
    position, then add it on top of the word embedding. The result is a
    single vector that carries both WHAT the word is and WHERE it sits.

    The sin/cos values use many different frequencies — like the hands of
    a clock at different speeds — so every position gets a unique pattern
    of values that the model can learn to read.
    """

    def __init__(self, d_model: int, seq_len: int, dropout: float):
        super().__init__()

        # d_model: how many dimensions each embedding vector has (e.g. 512).
        # We need this to know how many sin/cos values to compute per position.
        self.d_model = d_model

        # seq_len: the maximum number of tokens we will ever process.
        # We pre-compute positional encodings for ALL positions up to this
        # limit once at startup, then just slice out what we need at runtime.
        self.seq_len = seq_len

        # Dropout randomly zeroes some values during training with probability
        # `dropout` (e.g. 0.1 means 10% of values become 0).
        # This prevents the model from relying too heavily on any single value,
        # which helps it generalize better to sentences it has never seen.
        self.dropout = nn.Dropout(dropout)

        # --- Build the full positional encoding table (done once, at startup) ---

        # Create an empty table of shape (seq_len, d_model).
        # Rows = positions (0, 1, 2, ... seq_len-1)
        # Columns = embedding dimensions (0, 1, 2, ... d_model-1)
        # We will fill in each cell with the appropriate sin or cos value below.
        pe = torch.zeros(seq_len, d_model)

        # Create a column vector of position indices: [[0], [1], [2], ..., [seq_len-1]]
        # torch.arange gives a flat list [0, 1, 2, ...], shape (seq_len,)
        # .unsqueeze(1) inserts a new dimension making it shape (seq_len, 1) —
        # a column vector. This shape is needed so PyTorch can broadcast it
        # against div_term (which is a row) to fill the whole table in one step.
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)

        # Compute the frequencies for each dimension pair.
        # The original formula is:  1 / 10000^(2i / d_model)
        # We rewrite it as:         exp(2i * -log(10000) / d_model)
        # These are mathematically identical — the exp/log form is preferred
        # because computing large powers like 10000^(512/512) can cause
        # floating point errors, while exp() is numerically well-behaved.
        # torch.arange(0, d_model, 2) gives [0, 2, 4, ...] — the even indices,
        # one frequency value per sin/cos pair.
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        # Fill the even columns (0, 2, 4, ...) with sin values.
        # position * div_term broadcasts (seq_len, 1) × (d_model/2,)
        # into a (seq_len, d_model/2) grid — every position × every frequency.
        # pe[:, 0::2] means: all rows, every other column starting from 0.
        pe[:, 0::2] = torch.sin(position * div_term)

        # Fill the odd columns (1, 3, 5, ...) with cos values.
        # Same frequencies as sin — cos gives the "second reading" at each
        # frequency so that two positions that share a sin value can still
        # be told apart by their cos value.
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add a batch dimension at position 0: (seq_len, d_model) → (1, seq_len, d_model)
        # PyTorch processes data in batches of shape (batch_size, seq_len, d_model).
        # The 1 here means "one copy of pe" — PyTorch will automatically broadcast
        # (stretch) it to match however many sentences are in the batch, without
        # physically copying the data in memory.
        pe = pe.unsqueeze(0)

        # Register pe as a "buffer" rather than a plain attribute or a Parameter.
        # A buffer is a tensor that:
        #   - is NOT learned (no gradient, never updated by backpropagation)
        #   - IS saved when you call torch.save(model)
        #   - IS moved to GPU automatically when you call model.cuda()
        # This is the right choice for pe because it is fixed math, not a
        # learned weight — but we still want it to travel with the model.
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, actual_seq_len, d_model)
        # pe shape: (1, seq_len, d_model)  — precomputed for the maximum length

        # Slice pe down to the actual length of this input.
        # pe was built for the worst-case maximum length (e.g. 512 tokens),
        # but most sentences are shorter. x.shape[1] is the real length,
        # so pe[:, :x.shape[1], :] means:
        #   dim 0 →  :           keep the single batch copy (broadcasts to full batch)
        #   dim 1 →  :x.shape[1] keep only the first N rows (N = actual sentence length)
        #   dim 2 →  :           keep all d_model dimensions
        # .requires_grad_(False) explicitly tells PyTorch: do not track gradients
        # through pe during backpropagation — it is fixed, nothing to learn here.
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)

        # Apply dropout to the combined (word + position) embedding.
        # Randomly zeroes some values during training to improve generalization.
        # At inference time (model.eval()) dropout is automatically disabled.
        return self.dropout(x)

## 3. Layer Normalization

In [5]:
class LayerNormalization(nn.Module):
    """
    Layer Normalization — stabilizes training by rescaling each token's
    embedding to have mean 0 and std 1, then applying a learned rescaling.

    Why we need it: without normalization, values across the network can
    grow very large or very small, making training unstable and slow.
    This layer ensures values stay in a consistent range at every step.
    """

    def __init__(self, eps: float = 10**-6) -> None:
        super().__init__()

        # eps (epsilon): a tiny number added to the std before dividing.
        # This prevents division by zero in the rare case where all values
        # in a token's embedding are identical (std would be 0).
        # 10**-6 = 0.000001 — small enough to be harmless otherwise.
        self.eps = eps

        # alpha: a learnable scale factor, initialized to 1 (no scaling).
        # After normalizing, the model may need to re-scale the values —
        # alpha lets it learn how much. nn.Parameter tells PyTorch to
        # update this value during backpropagation, just like a weight.
        self.alpha = nn.Parameter(torch.ones(1))

        # bias: a learnable shift, initialized to 0 (no shift).
        # After normalizing, the model may need to shift the values up or
        # down — bias lets it learn by how much.
        # Together, alpha and bias let the model "undo" normalization if
        # needed, giving it full flexibility.
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        # Each token (one row of d_model values) is normalized independently.

        # Compute the mean of each token's embedding values.
        # dim=-1 means "average across the last dimension" (d_model),
        # so each token gets its own mean — not averaged across tokens.
        # keepdim=True keeps the shape as (batch, seq_len, 1) instead of
        # (batch, seq_len), so it can broadcast back against x correctly.
        mean = x.mean(dim=-1, keepdim=True)

        # Compute the standard deviation of each token's embedding values.
        # Same logic as mean: per-token, last dimension, shape kept.
        # std tells us how spread out the values are around the mean.
        std = x.std(dim=-1, keepdim=True)

        # Normalize: subtract mean so values are centered at 0,
        # then divide by (std + eps) so values are scaled to std of 1.
        # eps is added to std to avoid dividing by zero.
        # Then apply the learned rescaling: alpha stretches/shrinks the
        # values and bias shifts them up or down.
        # If the model learns alpha=1 and bias=0 it effectively does
        # nothing, preserving the normalized values as-is.
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

## 4. Feed Forward Network
It is a fully connected layer used both in encoder and decoder

$ FFN(x) = max(0, xW_1 + b_1)W_2 + b_2 $



In [6]:
class FeedForward(nn.Module):
    """
    Feed-Forward Block — a small neural network applied independently to
    each token's vector after the attention step.

    Why we need it: attention lets tokens communicate with each other and
    gather information. The feed-forward block then lets each token process
    and transform that gathered information on its own — no communication
    between tokens here, just individual computation.

    Structure: two linear layers with a ReLU activation in between.
        linear_1: expands each vector from d_model → d_ff  (wider = more capacity)
        ReLU:     introduces non-linearity so the network can learn complex patterns
        linear_2: compresses back from d_ff → d_model  (back to the expected shape)

    Typical sizes: d_model=512, d_ff=2048 (4× expansion), so each token's
    vector goes 512 → 2048 → 512 through this block.
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()

        # First linear layer: expands each token vector from d_model to d_ff.
        # nn.Linear(in, out) creates a learnable weight matrix of shape
        # (in, out) and a bias vector of shape (out,). During forward it
        # computes: output = input × weights + bias
        # The larger d_ff gives the network more intermediate neurons to
        # work with — more capacity to detect different patterns in the data.
        self.linear_1 = nn.Linear(d_model, d_ff)

        # Dropout randomly zeroes some neuron outputs during training with
        # probability `dropout` (e.g. 0.1 = 10% of values become 0).
        # This prevents the network from relying too heavily on any single
        # neuron, which helps it generalize to unseen data.
        # At inference time (model.eval()) dropout is automatically disabled.
        self.dropout = nn.Dropout(dropout)

        # Second linear layer: compresses back from d_ff to d_model.
        # This restores the original vector size so the output of this block
        # has the same shape as the input — required because the transformer
        # adds the input back (residual connection) right after this block.
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # x shape coming in: (batch_size, seq_len, d_model)
        # The same sequence of operations is applied to every token independently.

        # Step 1 — linear_1: project each token vector from d_model to d_ff.
        # Shape: (batch, seq_len, d_model) → (batch, seq_len, d_ff)
        # Each of the d_ff output values is a learned weighted sum of all
        # d_model input values — different neurons look for different patterns.
        x = self.linear_1(x)

        # Step 2 — ReLU: apply the activation function element-wise.
        # ReLU(x) = max(0, x): negative values become 0, positives pass through.
        # This is critical — without it, linear_1 followed by linear_2 would
        # just be one big linear transformation (linear × linear = linear),
        # no matter how large d_ff is. ReLU introduces non-linearity, letting
        # the network learn curved, complex relationships in the data.
        # Shape stays: (batch, seq_len, d_ff)
        x = torch.relu(x)

        # Step 3 — dropout: randomly zero some values during training.
        # Applied after ReLU so we drop activated neuron outputs.
        # Shape stays: (batch, seq_len, d_ff)
        x = self.dropout(x)

        # Step 4 — linear_2: project back from d_ff to d_model.
        # This compresses the expanded representation back to the original
        # size. The network has to summarize everything it computed in the
        # wider space into d_model values — forcing it to keep only what matters.
        # Shape: (batch, seq_len, d_ff) → (batch, seq_len, d_model)
        return self.linear_2(x)

        # NOTE: the original code writes all four steps in one line:
        #   return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))
        # This is identical — functions are just nested inside each other.
        # The expanded version above is easier to read and debug.

## 5. Multi-Head Attention

In [7]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention — lets every token look at every other token
    and decide how much to attend to each one, from multiple perspectives.

    Why we need it: to understand a word you often need context from other
    words. "it" needs to find its antecedent, a verb needs to find its
    subject, etc. Attention computes these relationships explicitly.

    Why multiple heads: a single attention pass can only look for one kind
    of relationship at a time. Multiple heads run in parallel, each free to
    specialize in a different type of relationship (syntax, coreference,
    proximity, etc.), giving the model much richer understanding.
    """

    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()

        # d_model: the size of each token's embedding vector (e.g. 512).
        self.d_model = d_model

        # h: number of attention heads running in parallel (e.g. 8).
        # Each head will work with a slice of d_model dimensions.
        self.h = h

        # Each head gets an equal share of the embedding dimensions.
        # d_model must divide evenly by h — if not, we can't split fairly.
        # The assert will crash with a clear message if this is violated,
        # catching a misconfiguration early rather than getting a cryptic
        # shape error later.
        assert d_model % h == 0, "d_model is not divisible by h"

        # d_k: how many dimensions each head works with.
        # Example: d_model=512, h=8 → d_k=64
        # Each head sees a 64-dimensional slice instead of the full 512.
        self.d_k = d_model // h

        # Three separate linear projections — one each for Query, Key, Value.
        # Even though q, k, v might be the same input (in self-attention),
        # these learned weight matrices project them into three different
        # spaces, letting the model learn different representations for
        # "what am I looking for" (Q), "what do I offer" (K), and
        # "what information do I carry" (V).
        # Shape of each: (d_model, d_model) — input and output size same.
        self.w_q = nn.Linear(d_model, d_model)  # projects input into query space
        self.w_k = nn.Linear(d_model, d_model)  # projects input into key space
        self.w_v = nn.Linear(d_model, d_model)  # projects input into value space

        # Final output projection: after all heads are concatenated back
        # together, w_o mixes and blends their outputs into one coherent
        # vector. This lets the model learn how to combine what different
        # heads found.
        self.w_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        """
        The core attention computation — static because it's pure math
        that doesn't need access to any learned weights (those were already
        applied before this method is called).

        @staticmethod means you can call this as MultiHeadAttention.attention(...)
        without needing an instance of the class. It's just a function that
        lives inside the class for organizational clarity.
        """

        # d_k: size of each head's query/key vectors.
        # We read it from the last dimension of query rather than self.d_k
        # so this method stays self-contained and reusable.
        d_k = query.shape[-1]

        # --- Step 1: compute raw attention scores ---
        # query shape:  (batch, h, seq_len, d_k)
        # key shape:    (batch, h, seq_len, d_k)
        #
        # key.transpose(-2, -1) flips the last two dimensions:
        #   (batch, h, seq_len, d_k) → (batch, h, d_k, seq_len)
        #
        # query @ key.transpose gives: (batch, h, seq_len, seq_len)
        # Entry [b, head, i, j] = dot product of token i's query
        # with token j's key = "how much should token i attend to token j?"
        #
        # Dividing by sqrt(d_k) prevents scores from getting too large,
        # which would cause softmax to become too "peaky" (one token gets
        # almost all attention, everything else is ignored).
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

        # --- Step 2: apply mask (optional) ---
        # The mask is a matrix of 1s (attend) and 0s (block).
        # Wherever mask == 0, we set the score to a huge negative number.
        # After softmax, e^(-1e9) ≈ 0, so those positions get ~zero weight
        # and are effectively invisible to the attending token.
        # Used for: hiding padding tokens, or hiding future tokens in decoder.
        if mask is not None:
            attention_scores.masked_fill_(mask == 0, -1e9)

        # --- Step 3: softmax — convert scores to probabilities ---
        # dim=-1 means softmax is applied across the last dimension (the keys),
        # so for each query token, its scores across all key tokens sum to 1.
        # These are the final attention weights: how much to attend to each token.
        attention_scores = attention_scores.softmax(dim=-1)
        # shape: (batch, h, seq_len, seq_len)

        # Optionally zero out some attention weights during training.
        # This encourages the model not to over-rely on any single token.
        if dropout is not None:
            attention_scores = dropout(attention_scores)

        # --- Step 4: collect values weighted by attention ---
        # attention_scores: (batch, h, seq_len, seq_len)
        # value:            (batch, h, seq_len, d_k)
        # result:           (batch, h, seq_len, d_k)
        #
        # For each token, this computes a weighted average of all value vectors,
        # where the weights come from attention_scores. Tokens with high scores
        # contribute more to the output. This is the "collected answer" —
        # information gathered from the most relevant other tokens.
        #
        # We also return attention_scores so they can be inspected/visualized
        # outside the model (useful for debugging and understanding what
        # the model learned to attend to).
        return (attention_scores @ value), attention_scores

    def forward(self, q, k, v, mask):
        # q, k, v shape: (batch_size, seq_len, d_model)
        # In self-attention (encoder), all three are the same input.
        # In cross-attention (decoder), q comes from the decoder and
        # k, v come from the encoder output.

        # --- Step 1: linear projections ---
        # Project q, k, v into their respective learned spaces.
        # Even if q == k == v (self-attention), these produce three
        # different tensors because w_q, w_k, w_v have different weights.
        # Shape stays: (batch, seq_len, d_model)
        query = self.w_q(q)
        key   = self.w_k(k)
        value = self.w_v(v)

        # --- Step 2: split into h heads ---
        # .view() reshapes the last dimension d_model into (h, d_k):
        #   (batch, seq_len, d_model) → (batch, seq_len, h, d_k)
        # .transpose(1, 2) swaps seq_len and h dimensions:
        #   (batch, seq_len, h, d_k) → (batch, h, seq_len, d_k)
        #
        # After this, dim 1 is the head index — PyTorch will treat each
        # head like an independent item in the batch, running all h heads
        # in parallel without any explicit loop.
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key   = key.view(key.shape[0],     key.shape[1],   self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        # --- Step 3: compute attention for all heads in parallel ---
        # x shape:                    (batch, h, seq_len, d_k)
        # self.attention_scores shape: (batch, h, seq_len, seq_len)
        # We save attention_scores as an attribute so they can be inspected
        # after the forward pass (useful for visualizing what the model attends to).
        x, self.attention_scores = MultiHeadAttention.attention(
            query, key, value, mask, self.dropout
        )

        # --- Step 4: reassemble all heads back into one tensor ---
        # .transpose(1, 2) swaps h and seq_len back:
        #   (batch, h, seq_len, d_k) → (batch, seq_len, h, d_k)
        # .contiguous() ensures the tensor's memory is laid out sequentially
        #   (transpose doesn't move data, just changes how it's indexed —
        #    .view() below requires the data to actually be contiguous in memory)
        # .view(..., self.h * self.d_k) merges the h and d_k dimensions:
        #   (batch, seq_len, h, d_k) → (batch, seq_len, d_model)
        #   where d_model = h * d_k  (just undoing the split from step 2)
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)

        # --- Step 5: final output projection ---
        # w_o mixes the concatenated head outputs together.
        # Each head found different things — w_o learns how to combine
        # those different perspectives into one coherent output vector.
        # Shape stays: (batch, seq_len, d_model)
        return self.w_o(x)

## 6. Residual Connections


In [8]:
class ResidualConnection(nn.Module):
    """
    Residual Connection — wraps any sublayer (attention or feed-forward)
    with three things: pre-normalization, dropout, and a skip connection
    that adds the original input back to the sublayer's output.

    Why we need it: transformers stack many layers on top of each other.
    Without residual connections, gradients vanish during backpropagation
    — they shrink to nearly zero before reaching the early layers, so
    those layers learn almost nothing. The skip connection creates a
    "highway" for gradients to travel backwards directly, bypassing the
    sublayer entirely and keeping gradient flow healthy throughout.

    A second benefit: the sublayer only needs to learn what CHANGES
    (the "residual" — hence the name), not the full representation from
    scratch. Learning small corrections is much easier than learning
    the whole thing every time.

    This same class is reused to wrap both attention and feed-forward
    blocks, keeping the code clean and consistent.
    """

    def __init__(self, dropout: float) -> None:
        super().__init__()

        # Dropout randomly zeroes some values during training.
        # Applied to the sublayer's output before adding x back,
        # which prevents the model from over-relying on specific neurons
        # in that sublayer. Automatically disabled at inference time.
        self.dropout = nn.Dropout(dropout)

        # LayerNormalization stabilizes the values before they enter
        # the sublayer. This is "pre-norm" — normalizing the input
        # rather than the output. It ensures the sublayer always
        # receives well-behaved values regardless of how large or
        # unstable x has become after accumulating across many layers.
        self.norm = LayerNormalization()

    def forward(self, x, sublayer):
        # x:        the token embeddings coming in, shape (batch, seq_len, d_model)
        # sublayer: a callable (function or nn.Module) passed in from outside.
        #           This will be either the attention block or the feed-forward
        #           block depending on which part of the encoder/decoder calls this.
        #           Passing functions as arguments is normal Python — functions are
        #           just objects, and this makes ResidualConnection reusable for
        #           any sublayer without changing any code here.

        # The full operation, inside-out:
        #
        #   1. self.norm(x)           — normalize x to stabilize values
        #   2. sublayer(...)          — pass normalized x through attention or FF
        #   3. self.dropout(...)      — randomly zero some outputs for regularization
        #   4. x + ...                — add the ORIGINAL x back (the skip connection)
        #
        # Step 4 is the key: the original x is preserved and added back unchanged.
        # This means:
        #   - Gradients can flow backwards directly through the "+" without
        #     touching the sublayer at all — solving the vanishing gradient problem
        #   - The sublayer only needs to learn corrections to x, not x itself —
        #     a much simpler learning task
        return x + self.dropout(sublayer(self.norm(x)))

## 7. Encoder Block

In [9]:
class EncoderBlock(nn.Module):
    """
    One Encoder Block — a single "round of thinking" that every token's
    vector passes through. Contains two stages, each wrapped in a
    residual connection:

    Stage 1 — Self-Attention: every token looks at every other token
              in the same sequence and gathers relevant context.
    Stage 2 — Feed-Forward: each token individually processes and
              transforms the information it just gathered.

    The output has the exact same shape as the input (batch, seq_len,
    d_model), so blocks can be stacked as many times as needed.
    """

    def __init__(
        self,
        self_attention: MultiHeadAttention,
        feed_forward: FeedForward,
        dropout: float
    ) -> None:
        super().__init__()

        # The MultiHeadAttention layer for stage 1.
        # Called "self" attention because q, k, v all come from
        # the same sequence — each token attends to itself and
        # all other tokens in the same sentence.
        self.self_attention = self_attention

        # The FeedForward layer for stage 2.
        # Applied independently to each token after attention
        # has finished gathering context.
        self.feed_forward = feed_forward

        # Two residual connections — one wrapping each stage.
        # nn.ModuleList (not a plain Python list) is required so
        # PyTorch knows these modules exist: their parameters are
        # included in training, saved with the model, and moved
        # to GPU automatically.
        # The list comprehension [... for _ in range(2)] just
        # creates two separate ResidualConnection instances.
        self.residual_connections = nn.ModuleList(
            [ResidualConnection(dropout) for _ in range(2)]
        )

    def forward(self, x, src_mask):
        # x shape:        (batch_size, seq_len, d_model)
        # src_mask shape: (batch_size, 1, 1, seq_len)
        #   src_mask blocks attention to padding tokens — dummy tokens
        #   added to make all sentences in the batch the same length.
        #   We don't want real tokens wasting attention on padding.

        # --- Stage 1: Self-Attention wrapped in residual connection ---
        #
        # We need to pass self_attention as a one-argument function to
        # ResidualConnection, but self_attention needs four arguments
        # (q, k, v, mask). The lambda solves this by "baking in" the
        # mask and the fact that q=k=v=x (self-attention), while
        # exposing only the single x argument that ResidualConnection
        # expects.
        #
        # Equivalent to writing:
        #   def sublayer(x):
        #       return self.self_attention(x, x, x, src_mask)
        #
        # After this line, each token's vector has been enriched with
        # context gathered from every other token in the sequence.
        x = self.residual_connections[0](
            x, lambda x: self.self_attention(x, x, x, src_mask)
        )

        # --- Stage 2: Feed-Forward wrapped in residual connection ---
        #
        # self.feed_forward already takes exactly one argument (x),
        # so no lambda is needed — it can be passed directly.
        #
        # After this line, each token has individually processed and
        # transformed the context it gathered in stage 1.
        x = self.residual_connections[1](x, self.feed_forward)

        # Return the enriched token vectors — same shape as input.
        # This output becomes the input to the next EncoderBlock.
        return x


class Encoder(nn.Module):
    """
    Full Encoder — runs the input through N EncoderBlocks in sequence,
    each one refining the token representations a little more, then
    applies a final normalization before passing the result onward.

    In the original transformer paper N=6. Each pass through a block
    gives tokens a richer, more contextual representation of their
    meaning within the sentence.
    """

    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()

        # The list of EncoderBlock layers to run in sequence.
        # Stored as nn.ModuleList so PyTorch tracks all parameters
        # inside every block for training, saving, and GPU transfer.
        self.layers = layers

        # A final LayerNormalization applied after all blocks.
        # After many layers of accumulation (each adding residuals),
        # values can drift in scale. This final norm stabilizes them
        # before the encoder output is used by the decoder.
        self.norm = LayerNormalization()

    def forward(self, x, mask):
        # x shape:    (batch_size, seq_len, d_model)
        # mask shape: (batch_size, 1, 1, seq_len)
        #   The same source mask is passed to every block so that
        #   self-attention never attends to padding tokens at any layer.

        # Pass x through each EncoderBlock one at a time.
        # The output of each block becomes the input to the next.
        # With each pass, tokens accumulate richer understanding of
        # their context within the sentence.
        for layer in self.layers:
            x = layer(x, mask)

        # Apply final normalization to the fully processed representations
        # before returning them. These vectors now encode deep contextual
        # meaning for every token and will be used by the decoder as the
        # source of cross-attention.
        return self.norm(x)

## 8. Decoder

In [10]:
class DecoderBlock(nn.Module):
    """
    One Decoder Block — similar to an EncoderBlock but with THREE stages
    instead of two. The extra stage is cross-attention, which is how the
    decoder "reads" the encoder's understanding of the source sentence
    while generating the target sentence (e.g. during translation).

    Stage 1 — Masked Self-Attention: the decoder tokens attend to each
              other, but with a causal mask — token at position 3 cannot
              see tokens at positions 4, 5, 6... This prevents the model
              from "cheating" by looking at future target words during
              training.

    Stage 2 — Cross-Attention: the decoder tokens attend to the ENCODER
              output. Queries come from the decoder (what am I looking
              for?), Keys and Values come from the encoder (what did the
              source sentence contain?). This is how meaning flows from
              source to target.

    Stage 3 — Feed-Forward: each token individually processes and
              transforms what it gathered from stages 1 and 2.
    """

    def __init__(
        self,
        self_attention: MultiHeadAttention,   # stage 1 — masked self-attention
        cross_attention: MultiHeadAttention,  # stage 2 — cross-attention with encoder
        feed_forward: FeedForward,            # stage 3 — per-token transformation
        dropout: float
    ):
        super().__init__()

        self.self_attention = self_attention
        self.cross_attention = cross_attention
        self.feed_forward = feed_forward

        # Three residual connections — one wrapping each stage.
        # NOTE: the original code has a bug here: nn.Module([...]) should
        # be nn.ModuleList([...]). nn.Module is the base class for all
        # layers and does not accept a list argument. nn.ModuleList is the
        # container that holds a list of modules and registers them all
        # with PyTorch so their parameters are tracked for training,
        # saving, and GPU transfer.
        self.residual_connections = nn.ModuleList(
            [ResidualConnection(dropout) for _ in range(3)]
        )

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        # x shape:              (batch_size, tgt_seq_len, d_model)
        #   — the target tokens processed so far (e.g. words generated so far)
        # encoder_output shape: (batch_size, src_seq_len, d_model)
        #   — the final enriched representations from the full encoder stack
        # src_mask:  blocks attention to padding tokens in the SOURCE sentence
        # tgt_mask:  causal mask — blocks each target token from attending to
        #            any token that comes AFTER it in the target sequence,
        #            preventing the model from seeing future words during training

        # --- Stage 1: Masked Self-Attention ---
        # Target tokens attend to each other, but only to tokens that came
        # BEFORE them (enforced by tgt_mask). This means when predicting
        # word 5, the model can only use words 1-4, never 6, 7, 8...
        # q = k = v = x because this is self-attention within the target sequence.
        # Lambda wraps the four-argument call into the single-argument interface
        # that ResidualConnection expects (same pattern as in EncoderBlock).
        x = self.residual_connections[0](
            x, lambda x: self.self_attention(x, x, x, tgt_mask)
        )

        # --- Stage 2: Cross-Attention ---
        # This is the bridge between encoder and decoder.
        # Queries (q) come from the decoder — each target token asks:
        #   "which parts of the source sentence are relevant to me?"
        # Keys (k) and Values (v) come from the encoder output — the
        #   encoder's full understanding of the source sentence.
        # src_mask ensures decoder tokens don't attend to source padding.
        # After this stage, each target token carries information gathered
        # from the most relevant parts of the source sentence.
        x = self.residual_connections[1](
            x, lambda x: self.cross_attention(x, encoder_output, encoder_output, src_mask)
        )

        # --- Stage 3: Feed-Forward ---
        # Each token individually processes and transforms the combined
        # information from self-attention and cross-attention.
        # self.feed_forward takes exactly one argument so no lambda needed.
        x = self.residual_connections[2](x, self.feed_forward)

        # Return enriched target token vectors — same shape as input.
        # This becomes the input to the next DecoderBlock in the stack.
        return x


class Decoder(nn.Module):
    """
    Full Decoder — runs the target sequence through N DecoderBlocks,
    passing the encoder output into every block so cross-attention can
    read the source sentence at each layer. Ends with a final
    LayerNormalization before the output is passed to the projection layer.

    The decoder is what actually generates output — at inference time it
    runs one token at a time, each time attending to all previously
    generated tokens and to the full encoder output.
    """

    def __init__(self, layers: nn.ModuleList) -> None:
        super().__init__()

        # The list of DecoderBlock layers to run in sequence.
        # nn.ModuleList ensures all parameters inside every block are
        # tracked by PyTorch for training, saving, and GPU transfer.
        self.layers = layers

        # Final LayerNormalization applied after all blocks — same
        # reasoning as in the Encoder: stabilizes accumulated values
        # before the output is used by the next component.
        self.norm = LayerNormalization()

    def forward(self, x, encoder_output, src_mask, tgt_mask):
        # x shape:              (batch_size, tgt_seq_len, d_model)
        # encoder_output shape: (batch_size, src_seq_len, d_model)
        # src_mask and tgt_mask are passed through to every block
        # so that cross-attention and self-attention always mask
        # the correct positions at every layer of the stack.

        # Pass x through each DecoderBlock in sequence.
        # encoder_output is passed to every block — each block gets
        # a fresh chance to cross-attend to the source sentence,
        # building progressively richer target representations.
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)

        # Final normalization before returning the decoded representations.
        # These vectors will be fed into the projection layer (a linear
        # layer + softmax) which converts them into probability distributions
        # over the vocabulary — one probability per possible next token.
        return self.norm(x)

## 9. Linear Layer

In [11]:
class LinearLayer(nn.Module):
    """
    Projection Layer — the final layer of the transformer. Converts the
    decoder's output vectors into a probability distribution over the
    entire vocabulary, giving one probability per possible next token.

    This is the layer that produces the actual answer to the question:
    "given everything the encoder and decoder have computed, what word
    should come next?"

    Two steps happen here:
        1. A linear projection expands each vector from d_model dimensions
           to vocab_size dimensions — one score (called a "logit") per word
           in the vocabulary.
        2. log_softmax converts those raw scores into log-probabilities so
           the model can be trained with a standard loss function.
    """

    def __init__(self, d_model: int, vocab_size: int) -> None:
        super().__init__()

        # A linear layer that projects each token vector from d_model
        # dimensions up to vocab_size dimensions.
        # Example: d_model=512, vocab_size=30000
        #   input:  (batch, seq_len, 512)
        #   output: (batch, seq_len, 30000)
        # Each of the 30000 output values is a raw score (logit) for one
        # word in the vocabulary — higher score means the model thinks
        # that word is more likely to come next.
        self.proj = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)

        # Step 1 — linear projection: map each token vector to a score
        # for every word in the vocabulary.
        # Shape: (batch, seq_len, d_model) → (batch, seq_len, vocab_size)
        logits = self.proj(x)

        # Step 2 — log_softmax: convert the raw scores into log-probabilities.
        #
        # Plain softmax would give:
        #   softmax(score_i) = e^score_i / sum(e^score_j for all j)
        # This turns any set of scores into positive values that sum to 1.0
        # — a proper probability distribution over the vocabulary.
        #
        # We use the LOG of softmax instead for two reasons:
        #
        #   1. Numerical stability: softmax alone computes e^score, which
        #      can explode to infinity for large scores. Taking the log
        #      keeps values in a safe numerical range.
        #
        #   2. Loss function compatibility: the standard loss used to train
        #      transformers is Negative Log-Likelihood (NLL). It expects
        #      log-probabilities as input, so returning log_softmax here
        #      means the loss function can be applied directly without an
        #      extra conversion step.
        #
        # dim=-1 means softmax is computed across the last dimension
        # (vocab_size), so for each token position the scores across all
        # vocabulary words sum to 1.0 in probability space (or 0.0 in
        # log space, since log(1) = 0).
        #
        # Output values are in range (-inf, 0] — log of a probability,
        # which is always between 0 and 1, is always <= 0.
        # The word with the score closest to 0 is the most likely next token.
        return torch.log_softmax(logits, dim=-1)

## 9. Transformer Block

In [ ]:
class Transformer(nn.Module):
    """
    The full Transformer model — assembles every component you have built
    so far into one complete system capable of sequence-to-sequence tasks
    like translation, summarization, or question answering.

    The architecture has two sides:
        Encoder side: reads and deeply understands the SOURCE sequence
                      (e.g. an English sentence to be translated)
        Decoder side: generates the TARGET sequence one token at a time,
                      attending to both what it has generated so far AND
                      to the encoder's understanding of the source

    Interestingly, encode(), decode(), and project() are kept as THREE
    SEPARATE methods rather than one single forward(). This is deliberate:
    during inference you only need to run encode() ONCE for a source
    sentence, then call decode() + project() repeatedly for each new
    token generated — reusing the encoder output every time without
    recomputing it. This makes generation much more efficient.
    """

    def __init__(
        self,
        encoder: Encoder,
        decoder: Decoder,
        src_embed: InputEmbeddings,    # embedding table for the source language
        tgt_embed: InputEmbeddings,    # embedding table for the target language
        src_pos: PositionalEncoding,   # positional encoding for source sequence
        tgt_pos: PositionalEncoding,   # positional encoding for target sequence
        projection_layer: LinearLayer  # final vocab projection
    ):
        super().__init__()

        # Source and target get SEPARATE embedding tables because they
        # are typically different languages with different vocabularies.
        # English "cat" and French "chat" are different token IDs in
        # different vocabularies — each needs its own lookup table.
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed

        # Positional encodings are also kept separate for source and
        # target in case the two sequences have different maximum lengths,
        # though the sin/cos computation itself is the same formula.
        self.src_pos = src_pos
        self.tgt_pos = tgt_pos

        # The encoder stack — N EncoderBlocks that build a deep
        # contextual understanding of the source sequence.
        self.encoder = encoder

        # The decoder stack — N DecoderBlocks that generate the target
        # sequence, attending to itself and to the encoder output.
        self.decoder = decoder

        # The final projection layer that converts decoder output vectors
        # into probability distributions over the target vocabulary.
        self.projection_layer = projection_layer

    def encode(self, src, src_mask):
        # src shape:      (batch_size, src_seq_len)  — raw source token IDs
        # src_mask shape: (batch_size, 1, 1, src_seq_len)

        # Step 1 — token IDs → dense embedding vectors.
        # Each integer token ID is replaced by its learned d_model-dimensional
        # vector from the source embedding lookup table.
        # Shape: (batch, src_seq_len) → (batch, src_seq_len, d_model)
        src = self.src_embed(src)

        # Step 2 — inject positional information.
        # Adds sin/cos positional encoding so the encoder knows the order
        # of tokens (transformers have no built-in sense of sequence order).
        # Shape stays: (batch, src_seq_len, d_model)
        src = self.src_pos(src)

        # Step 3 — run through the full encoder stack.
        # N EncoderBlocks progressively build rich contextual representations.
        # src_mask prevents attention to padding tokens in the source.
        # Shape stays: (batch, src_seq_len, d_model)
        # This output is returned and reused by every decode() call —
        # computed once, referenced many times during generation.
        return self.encoder(src, src_mask)

    def decode(self, encoder_output, src_mask, tgt, tgt_mask):
        # encoder_output shape: (batch_size, src_seq_len, d_model)
        #   — the fixed output from encode(), reused on every decode() call
        # src_mask:   blocks cross-attention to source padding tokens
        # tgt shape:  (batch_size, tgt_seq_len)  — target token IDs generated so far
        # tgt_mask:   causal mask — prevents each target token from attending
        #             to future target tokens (no peeking ahead)

        # Step 1 — token IDs → dense embedding vectors, using the TARGET
        # embedding table (separate from source — different vocabulary).
        # Shape: (batch, tgt_seq_len) → (batch, tgt_seq_len, d_model)
        tgt = self.tgt_embed(tgt)

        # Step 2 — inject positional information for the target sequence.
        # Shape stays: (batch, tgt_seq_len, d_model)
        tgt = self.tgt_pos(tgt)

        # Step 3 — run through the full decoder stack.
        # Each DecoderBlock performs:
        #   - masked self-attention over target tokens generated so far
        #   - cross-attention over encoder_output (reading the source)
        #   - feed-forward transformation
        # Both masks are passed to every block in the stack.
        # Shape stays: (batch, tgt_seq_len, d_model)
        return self.decoder(tgt, encoder_output, src_mask, tgt_mask)

    def project(self, x):
        # x shape: (batch_size, tgt_seq_len, d_model)
        #   — the decoder output for the current generation step

        # Project from d_model dimensions to vocab_size dimensions,
        # then apply log_softmax to get log-probabilities over the
        # entire target vocabulary.
        # Shape: (batch, tgt_seq_len, d_model) → (batch, tgt_seq_len, vocab_size)
        #
        # In practice during inference you typically only care about the
        # LAST position — the probability distribution for the NEXT token:
        #   next_token_logprobs = project(decoder_out)[:, -1, :]
        # The token with the highest log-probability is selected as the
        # next generated word (greedy decoding), or sampled from the
        # distribution (stochastic decoding).
        return self.projection_layer(x)

## 10. Build Transformer

In [ ]:
def build_transformer(
    src_vocab_size: int,   # number of unique tokens in the source language vocabulary
    tgt_vocab_size: int,   # number of unique tokens in the target language vocabulary
    src_seq_len: int,      # maximum source sequence length (e.g. 512 tokens)
    tgt_seq_len: int,      # maximum target sequence length (e.g. 512 tokens)
    d_model: int = 512,    # embedding dimension — size of every token vector throughout the model
    N: int = 6,            # number of encoder AND decoder blocks to stack
    h: int = 8,            # number of attention heads in every MultiHeadAttention layer
    dropout: float = 0.1,  # dropout probability used throughout (10% of values zeroed during training)
    d_ff: int = 2048       # inner dimension of feed-forward blocks (typically 4 × d_model)
) -> Transformer:
    """
    Factory function — instantiates and wires together every component you
    have built so far into one complete, ready-to-train Transformer model.

    A factory function is just a regular function whose job is to construct
    and return a complex object. Rather than building the transformer by hand
    every time (creating embeddings, then blocks, then encoder, then decoder...),
    you call this once with your hyperparameters and get back a fully assembled,
    weight-initialized model.

    The hyperparameters here match the "base" model from the original
    "Attention Is All You Need" paper (Vaswani et al., 2017):
        d_model=512, N=6, h=8, d_ff=2048, dropout=0.1
    """

    # --- Embedding layers ---
    # Source and target get completely separate embedding tables because
    # they have different vocabularies (e.g. English vs French).
    # Each table is a (vocab_size × d_model) matrix of learnable vectors —
    # one row per token, learned during training.
    src_embed = InputEmbeddings(d_model, src_vocab_size)
    tgt_embed = InputEmbeddings(d_model, tgt_vocab_size)

    # --- Positional encoding layers ---
    # Pre-computes the sin/cos position table up to the maximum sequence
    # length for each side. Kept separate in case src and tgt have
    # different maximum lengths.
    src_pos = PositionalEncoding(d_model, src_seq_len, dropout)
    tgt_pos = PositionalEncoding(d_model, tgt_seq_len, dropout)

    # --- Build N independent EncoderBlocks ---
    # Each block gets its OWN freshly instantiated attention and feed-forward
    # layers with separate, independently learned weights. This is important —
    # if all blocks shared the same weights they would all learn the same
    # transformation, defeating the purpose of stacking them.
    encoder_blocks = []
    for _ in range(N):
        # One MultiHeadAttention for self-attention within the source sequence.
        encoder_self_attention_block = MultiHeadAttention(d_model, h, dropout)
        # One FeedForward for per-token transformation after attention.
        feed_forward_block = FeedForward(d_model, d_ff, dropout)
        # Wire them together into one complete encoder block.
        encoder_block = EncoderBlock(
            encoder_self_attention_block, feed_forward_block, dropout
        )
        encoder_blocks.append(encoder_block)

    # --- Build N independent DecoderBlocks ---
    # Each decoder block needs THREE sub-layers (vs two for encoder):
    # self-attention, cross-attention, and feed-forward — all independent
    # across blocks for the same reason as the encoder.
    decoder_blocks = []
    for _ in range(N):
        # Self-attention over the target tokens generated so far
        # (masked so future tokens cannot be seen).
        decoder_self_attention_block = MultiHeadAttention(d_model, h, dropout)
        # Cross-attention — queries from decoder, keys/values from encoder.
        # This is the bridge that lets the decoder read the source sentence.
        decoder_cross_attention_block = MultiHeadAttention(d_model, h, dropout)
        # Feed-forward for per-token transformation after both attention stages.
        feed_forward_block = FeedForward(d_model, d_ff, dropout)
        # Wire all three together into one complete decoder block.
        decoder_block = DecoderBlock(
            decoder_self_attention_block,
            decoder_cross_attention_block,
            feed_forward_block,
            dropout
        )
        decoder_blocks.append(decoder_block)

    # --- Assemble encoder and decoder stacks ---
    # nn.ModuleList wraps the plain Python lists so PyTorch tracks every
    # parameter inside every block for training, saving, and GPU transfer.
    encoder = Encoder(nn.ModuleList(encoder_blocks))
    decoder = Decoder(nn.ModuleList(decoder_blocks))

    # --- Projection layer ---
    # The final linear + log_softmax layer that converts decoder output
    # vectors into log-probability distributions over the TARGET vocabulary.
    # Uses tgt_vocab_size (not src) because we are predicting target tokens.
    projection_layer = LinearLayer(d_model, tgt_vocab_size)

    # --- Assemble the full Transformer ---
    # Wires every component together into the single model object that
    # exposes encode(), decode(), and project() methods.
    transformer = Transformer(
        encoder, decoder,
        src_embed, tgt_embed,
        src_pos, tgt_pos,
        projection_layer
    )

    # --- Weight initialization: Xavier uniform ---
    # All learnable weight matrices with more than 1 dimension get
    # initialized using Xavier uniform initialization (also called
    # Glorot uniform, after its author).
    #
    # Why not just leave weights at their default random values?
    # If weights start too large, activations explode through the layers.
    # If weights start too small, activations vanish. Xavier initialization
    # carefully sets the initial scale of weights based on the size of
    # the layer (number of inputs and outputs), keeping the variance of
    # activations roughly consistent from layer to layer at the start of
    # training. This gives the optimizer a much healthier starting point.
    #
    # p.dim() > 1 skips 1D tensors (bias vectors) — biases are left at
    # their default initialization (zeros) since Xavier is designed for
    # weight matrices, not bias vectors.
    for p in transformer.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    # NOTE: the original code has a bug here — it returns None because
    # the return statement is empty. It should be:
    #   return transformer
    return transformer